# Quantization & Edge Deployment

This notebook demonstrates model quantization for edge device deployment.

**Use cases:**
- Raspberry Pi
- Jetson Nano
- IoT gateways
- Microcontrollers with Python
- Docker microservices

**Quantization options:**
- `float16`: 50% memory reduction, minimal accuracy loss
- `int8`: 75% memory reduction, some accuracy trade-off

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '../src')

from neura_command import (
    Sequential, Dense, ReLU, Sigmoid, Dropout, Softmax,
    Adam, MSELoss, CrossEntropyLoss,
    InferenceModel, quantize_model, QuantizationConfig,
)

np.random.seed(42)

## 1. Train a Model

In [ ]:
# Generate training data
n_samples = 1000
n_features = 20
n_classes = 5

X = np.random.randn(n_samples, n_features).astype(np.float32)
y_labels = np.random.randint(0, n_classes, n_samples)
y = np.eye(n_classes)[y_labels].astype(np.float32)

# Split
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
y_test_labels = y_labels[split:]

print(f"Training: {len(X_train)} samples")
print(f"Testing:  {len(X_test)} samples")

In [ ]:
# Create and train model
model = Sequential(
    layers=[
        Dense(n_features, 128),
        ReLU(),
        Dense(128, 64),
        ReLU(),
        Dense(64, 32),
        ReLU(),
        Dense(32, n_classes),
        Softmax(),
    ],
    optimizer=Adam(learning_rate=0.01),
    loss=CrossEntropyLoss(),
)

print(model.summary())

In [ ]:
# Train
model.train(X_train, y_train, epochs=30, batch_size=32, verbose=False)

# Evaluate
preds = model.predict(X_test)
pred_labels = np.argmax(preds, axis=1)
accuracy = np.mean(pred_labels == y_test_labels) * 100

print(f"Original model accuracy: {accuracy:.1f}%")

## 2. Create Inference Model

`InferenceModel` is a lightweight wrapper optimized for prediction only.
No backward pass, no optimizer, no training code.

In [ ]:
# Create inference model (no quantization)
inference = InferenceModel.from_sequential(model)

print(inference.info())

In [ ]:
# Verify predictions match
inf_preds = inference.predict(X_test)
inf_labels = np.argmax(inf_preds, axis=1)
inf_accuracy = np.mean(inf_labels == y_test_labels) * 100

print(f"Inference model accuracy: {inf_accuracy:.1f}%")
print(f"Predictions match: {np.allclose(preds, inf_preds)}")

## 3. Float16 Quantization

Half-precision reduces memory by 50% with minimal accuracy loss.

In [ ]:
# Create float16 inference model
inference_fp16 = InferenceModel.from_sequential(model, quantize='float16')

print(inference_fp16.info())

In [ ]:
# Compare memory usage
memory_fp32 = inference.memory_bytes
memory_fp16 = inference_fp16.memory_bytes

print(f"FP32 memory: {memory_fp32 / 1024:.1f} KB")
print(f"FP16 memory: {memory_fp16 / 1024:.1f} KB")
print(f"Reduction:   {(1 - memory_fp16/memory_fp32) * 100:.1f}%")

In [ ]:
# Check accuracy
fp16_preds = inference_fp16.predict(X_test)
fp16_labels = np.argmax(fp16_preds, axis=1)
fp16_accuracy = np.mean(fp16_labels == y_test_labels) * 100

print(f"Original accuracy: {accuracy:.1f}%")
print(f"FP16 accuracy:     {fp16_accuracy:.1f}%")
print(f"Accuracy loss:     {accuracy - fp16_accuracy:.2f}%")

## 4. Int8 Quantization

8-bit quantization reduces memory by 75% with some accuracy trade-off.

In [ ]:
# Create int8 inference model
inference_int8 = InferenceModel.from_sequential(model, quantize='int8')

print(inference_int8.info())

In [ ]:
# Compare memory usage
memory_int8 = inference_int8.memory_bytes

print(f"FP32 memory: {memory_fp32 / 1024:.1f} KB")
print(f"Int8 memory: {memory_int8 / 1024:.1f} KB")
print(f"Reduction:   {(1 - memory_int8/memory_fp32) * 100:.1f}%")

In [ ]:
# Check accuracy
int8_preds = inference_int8.predict(X_test)
int8_labels = np.argmax(int8_preds, axis=1)
int8_accuracy = np.mean(int8_labels == y_test_labels) * 100

print(f"Original accuracy: {accuracy:.1f}%")
print(f"Int8 accuracy:     {int8_accuracy:.1f}%")
print(f"Accuracy loss:     {accuracy - int8_accuracy:.2f}%")

## 5. Comparison Summary

In [ ]:
import matplotlib.pyplot as plt

# Summary data
formats = ['FP32\n(Original)', 'FP16\n(Half)', 'Int8\n(Quantized)']
memories = [memory_fp32 / 1024, memory_fp16 / 1024, memory_int8 / 1024]
accuracies = [accuracy, fp16_accuracy, int8_accuracy]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Memory comparison
colors = ['steelblue', 'orange', 'green']
ax1.bar(formats, memories, color=colors)
ax1.set_ylabel('Memory (KB)')
ax1.set_title('Memory Usage')
for i, (fmt, mem) in enumerate(zip(formats, memories)):
    ax1.text(i, mem + 0.5, f'{mem:.1f} KB', ha='center')

# Accuracy comparison
ax2.bar(formats, accuracies, color=colors)
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Model Accuracy')
ax2.set_ylim(0, 100)
for i, (fmt, acc) in enumerate(zip(formats, accuracies)):
    ax2.text(i, acc + 1, f'{acc:.1f}%', ha='center')

plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"{'Format':<12} {'Memory':<12} {'Accuracy':<12} {'Memory Saved':<12}")
print("-" * 48)
print(f"{'FP32':<12} {memory_fp32/1024:.1f} KB{'':<6} {accuracy:.1f}%{'':<8} -")
print(f"{'FP16':<12} {memory_fp16/1024:.1f} KB{'':<6} {fp16_accuracy:.1f}%{'':<8} {(1-memory_fp16/memory_fp32)*100:.0f}%")
print(f"{'Int8':<12} {memory_int8/1024:.1f} KB{'':<6} {int8_accuracy:.1f}%{'':<8} {(1-memory_int8/memory_fp32)*100:.0f}%")

## 6. Load from File with Quantization

In [ ]:
import tempfile
import os

# Save the model
with tempfile.TemporaryDirectory() as tmpdir:
    model_path = os.path.join(tmpdir, 'model')
    model.save(model_path)
    
    print("Loading model with different quantization levels...")
    
    # Load without quantization
    inf_fp32 = InferenceModel.from_file(model_path)
    print(f"FP32: {inf_fp32.memory_bytes / 1024:.1f} KB")
    
    # Load with FP16
    inf_fp16 = InferenceModel.from_file(model_path, quantize='float16')
    print(f"FP16: {inf_fp16.memory_bytes / 1024:.1f} KB")
    
    # Load with Int8
    inf_int8 = InferenceModel.from_file(model_path, quantize='int8')
    print(f"Int8: {inf_int8.memory_bytes / 1024:.1f} KB")

## 7. Inference Speed Comparison

In [ ]:
import time

# Benchmark function
def benchmark(model, X, n_runs=100):
    # Warmup
    for _ in range(10):
        _ = model.predict(X)
    
    # Timed runs
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = model.predict(X)
    elapsed = time.perf_counter() - start
    
    return elapsed / n_runs * 1000  # ms per inference

# Run benchmarks
X_bench = X_test[:100]  # 100 samples

time_fp32 = benchmark(inference, X_bench)
time_fp16 = benchmark(inference_fp16, X_bench)
time_int8 = benchmark(inference_int8, X_bench)

print(f"Inference time (100 samples):")
print(f"  FP32: {time_fp32:.3f} ms")
print(f"  FP16: {time_fp16:.3f} ms")
print(f"  Int8: {time_int8:.3f} ms")

## 8. Edge Deployment Example

Here's how you would deploy to a Raspberry Pi or similar device:

In [ ]:
# Example edge deployment code
deployment_code = '''
# edge_inference.py - Minimal code for edge deployment

import numpy as np
from neura_command import InferenceModel

# Load model with quantization for minimal memory
model = InferenceModel.from_file('model', quantize='float16')

print(f"Model loaded: {model.memory_bytes / 1024:.1f} KB")

# Run inference
def predict(sensor_data):
    """Make prediction from sensor data."""
    # Ensure correct shape and dtype
    x = np.array(sensor_data, dtype=np.float32).reshape(1, -1)
    
    # Get prediction
    output = model.predict(x)
    
    return int(np.argmax(output))

# Example usage
if __name__ == '__main__':
    # Simulate sensor reading
    sensor_reading = [0.1, 0.5, -0.3, 0.8, ...]  # 20 features
    
    prediction = predict(sensor_reading)
    print(f"Predicted class: {prediction}")
'''

print(deployment_code)

## Summary

### Quantization Options

| Format | Memory | Accuracy | Use Case |
|--------|--------|----------|----------|
| **FP32** | 100% | Best | Training, desktop inference |
| **FP16** | ~50% | Very Good | Jetson Nano, GPU inference |
| **Int8** | ~25% | Good | Raspberry Pi, microcontrollers |

### Deployment Steps

1. **Train** on powerful hardware
2. **Save** model: `model.save('path')`
3. **Transfer** files to edge device
4. **Load with quantization**: `InferenceModel.from_file('path', quantize='float16')`
5. **Run inference**: `model.predict(x)`

### NeuraCommand Advantages for Edge

- **Minimal dependencies**: Just NumPy
- **Small footprint**: ~50KB library code
- **No CUDA required**: Works on any Python device
- **Safe serialization**: No pickle security risks